# Etapa 3 — Transfer Learning com TorchVision

Este notebook registra a etapa de transfer learning para classificação binária de gatos e cães. Ele compara ResNet18, EfficientNet-B0 e ConvNeXt-Tiny com pesos ImageNet oficiais, preservando a mesma divisão de dataset e o mesmo contrato de métricas usados na CNN construída do zero.

O código de treinamento não foi copiado para dentro do notebook: este artefato chama os módulos de produção em `src/cnn_cats_dogs/`, evitando que a versão explicada e a versão executada virem projetos diferentes.


## 1. Pergunta experimental

A pergunta central é como a capacidade do backbone e o custo computacional afetam a generalização quando há somente 300 imagens de treino. O protocolo compara três famílias pré-treinadas:

- **ResNet18:** baseline residual clássico;
- **EfficientNet-B0:** alternativa compacta e eficiente;
- **ConvNeXt-Tiny:** alternativa de maior capacidade.

A seleção usa validação. O teste final é executado apenas depois de congelar arquiteturas, seeds, augmentations e hiperparâmetros.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from cnn_cats_dogs.transfer_config import TransferTrainingConfig
from cnn_cats_dogs.transfer_data import build_transfer_data_bundle, build_transfer_transforms
from cnn_cats_dogs.transfer_engine import run_transfer_training
from cnn_cats_dogs.transfer_models import transfer_model_ids, transfer_model_metadata

print(f'Raiz do projeto: {ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Integridade do dataset

Métricas próximas de 100% exigem verificação de integridade antes de virar conclusão. A auditoria procura duplicatas byte-a-byte entre splits e candidatos perceptualmente semelhantes. A ausência de duplicata exata deve ser registrada junto ao relatório.


In [ ]:
AUDIT_DIR = ROOT / 'runs' / 'dataset_audit'
audit_summary_path = AUDIT_DIR / 'summary.json'

if audit_summary_path.is_file():
    with audit_summary_path.open(encoding='utf-8') as file:
        audit_summary = json.load(file)
    display(pd.Series({
        'images_by_split': audit_summary['images_by_split'],
        'exact_cross_split_pairs': audit_summary['exact_cross_split_pairs'],
        'exact_cross_split_label_conflicts': audit_summary['exact_cross_split_label_conflicts'],
        'near_cross_split_candidate_pairs': audit_summary['near_cross_split_candidate_pairs'],
        'near_duplicate_dhash_threshold': audit_summary['near_duplicate_dhash_threshold'],
    }, name='Auditoria do dataset').to_frame())
else:
    print('Auditoria ainda não encontrada. Execute:')
    print('python3 scripts/audit_dataset.py --data-dir dataset --output-dir runs/dataset_audit --dhash-threshold 4')


## 3. Backbones e preprocessamento

Cada conjunto de pesos possui normalização, resize e interpolação próprios. Validação e teste usam diretamente `weights.transforms()`. No treino, crop/escala, flip, jitter e erasing são aplicados antes da normalização esperada pelos pesos ImageNet.


In [ ]:
model_table = pd.DataFrame([transfer_model_metadata(identifier) for identifier in transfer_model_ids()])
display(model_table[[
    'display_name', 'identifier', 'parameters_millions_reference',
    'flops_giga_reference', 'final_stage', 'weights'
]])


In [ ]:
DATA_DIR = ROOT / 'dataset'
POSITIVE_CLASS = 'dogs'
TRAIN_ARCHITECTURE = 'convnext_tiny'
RUN_TRAINING = False
REPRODUCTION_DIR = ROOT / 'runs' / 'transfer_reproduction' / TRAIN_ARCHITECTURE / 'seed_42'

config = TransferTrainingConfig(
    data_dir=DATA_DIR,
    output_dir=REPRODUCTION_DIR,
    architecture=TRAIN_ARCHITECTURE,
    positive_class=POSITIVE_CLASS,
    head_epochs=12,
    finetune_epochs=20,
    batch_size=32,
    head_learning_rate=1e-3,
    finetune_learning_rate=1e-4,
    weight_decay=1e-4,
    num_workers=8,
    seed=42,
    device='auto',
    use_amp=True,
    evaluate_test=False,
)
config.validate()
config


In [ ]:
preview_transforms = build_transfer_transforms(TRAIN_ARCHITECTURE)
print('Transform de avaliação oficial:')
print(preview_transforms.metadata['evaluation_transform'])
print('Augmentations de treino:', preview_transforms.metadata['train_augmentations'])

bundle = build_transfer_data_bundle(config)
print('Classes [0, 1]:', bundle.class_names)
print('Tamanhos:', bundle.sizes)
print('Contagens:', bundle.class_counts)

images, labels = next(iter(bundle.train_loader))
mean = torch.tensor(preview_transforms.metadata['normalization_mean']).view(3, 1, 1)
std = torch.tensor(preview_transforms.metadata['normalization_std']).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label in zip(axes.flat, images[:8], labels[:8]):
    visible = (image.cpu() * std + mean).clamp(0, 1)
    axis.imshow(visible.permute(1, 2, 0))
    axis.set_title(bundle.class_names[int(label)])
    axis.axis('off')
fig.suptitle(f'Amostras com augmentation para {TRAIN_ARCHITECTURE}')
fig.tight_layout()


## 4. Fine-tuning em duas fases

**Fase 1, adaptação da cabeça:** todos os parâmetros do backbone ficam congelados e somente o classificador binário novo é treinado.

**Fase 2, fine-tuning parcial:** o melhor checkpoint da primeira fase é recarregado; apenas o último estágio visual e o classificador são liberados com learning rate menor.

A decisão final usa a menor loss de validação. O checkpoint selecionado pode pertencer a qualquer uma das duas fases.


In [ ]:
if RUN_TRAINING:
    artifacts = run_transfer_training(config)
    print('Treino concluído:', artifacts.summary_path)
    print('Checkpoint escolhido por validação:', artifacts.best_checkpoint)
else:
    print('Treino desativado. O notebook lê as execuções consolidadas nas próximas células.')


## 5. Comparação por validação

Os três modelos foram avaliados com as seeds 42, 73 e 101. A tabela é ordenada pela média da melhor loss de validação. Nenhuma decisão de arquitetura foi tomada a partir do conjunto de teste.


In [ ]:
COMPARISON_DIR = ROOT / 'runs' / 'transfer_comparison'
comparison_runs_path = COMPARISON_DIR / 'comparison_runs.csv'
comparison_summary_path = COMPARISON_DIR / 'comparison_summary.csv'

if comparison_runs_path.is_file():
    comparison_runs = pd.read_csv(comparison_runs_path)
    display(comparison_runs.sort_values(['architecture', 'seed']))

    figure, axis = plt.subplots(figsize=(8, 4.5))
    for architecture, frame in comparison_runs.groupby('architecture'):
        axis.plot(frame['seed'], frame['best_validation_loss'], marker='o', label=architecture)
    axis.set_xlabel('Seed')
    axis.set_ylabel('Melhor loss de validação')
    axis.set_title('Estabilidade da seleção por validação')
    axis.legend()
    axis.grid(alpha=0.25)
    figure.tight_layout()

if comparison_summary_path.is_file():
    comparison_summary = pd.read_csv(comparison_summary_path)
    display(comparison_summary)
else:
    print('Resumo de comparação não encontrado:', comparison_summary_path)


## 6. Resultados finais no teste

Depois que o protocolo foi congelado, os nove checkpoints pré-definidos pela combinação de três modelos e três seeds foram avaliados no teste. A média entre seeds mede estabilidade de treinamento no mesmo split final; não deve ser lida como se houvesse 300 imagens de teste independentes.


In [ ]:
FINAL_TEST_DIR = ROOT / 'runs' / 'transfer_final_test'

def load_final_test_results(root: Path) -> pd.DataFrame:
    rows = []
    for summary_path in sorted(root.glob('**/artifacts/evaluation_summary.json')):
        with summary_path.open(encoding='utf-8') as file:
            payload = json.load(file)
        metrics = payload['test_metrics']
        run_dir = summary_path.parents[1]
        rows.append({
            'architecture': payload['architecture']['identifier'],
            'display_name': payload['architecture']['display_name'],
            'seed': run_dir.name.replace('seed_', ''),
            'loss': metrics['loss'],
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'run_dir': str(run_dir),
        })
    return pd.DataFrame(rows)

test_runs = load_final_test_results(FINAL_TEST_DIR)
if test_runs.empty:
    print('Resultados finais não encontrados em:', FINAL_TEST_DIR)
else:
    test_runs['seed'] = pd.to_numeric(test_runs['seed'])
    display(test_runs.sort_values(['architecture', 'seed']))


In [ ]:
if not test_runs.empty:
    metric_columns = ['loss', 'accuracy', 'precision', 'recall', 'f1']
    test_summary = test_runs.groupby('display_name')[metric_columns].agg(['mean', 'std'])
    display(test_summary)

    figure, axis = plt.subplots(figsize=(8, 4.5))
    by_model = test_runs.groupby('display_name')['accuracy'].mean().sort_values(ascending=False)
    axis.bar(by_model.index, by_model.values)
    axis.set_ylim(0.9, 1.0)
    axis.set_ylabel('Acurácia média no teste')
    axis.set_title('Acurácia média por backbone')
    axis.grid(axis='y', alpha=0.25)
    figure.tight_layout()


In [ ]:
def error_summary(root: Path) -> pd.DataFrame:
    rows = []
    for prediction_path in sorted(root.glob('**/artifacts/test_predictions.csv')):
        predictions = pd.read_csv(prediction_path)
        run_dir = prediction_path.parents[1]
        false_positive = int(((predictions['target'] == 0) & (predictions['prediction'] == 1)).sum())
        false_negative = int(((predictions['target'] == 1) & (predictions['prediction'] == 0)).sum())
        rows.append({
            'architecture': run_dir.parent.name,
            'seed': run_dir.name.replace('seed_', ''),
            'false_positive_cats_as_dogs': false_positive,
            'false_negative_dogs_as_cats': false_negative,
        })
    return pd.DataFrame(rows)

errors = error_summary(FINAL_TEST_DIR)
if not errors.empty:
    errors['seed'] = pd.to_numeric(errors['seed'])
    display(errors.sort_values(['architecture', 'seed']))


## 7. Conclusão

ConvNeXt-Tiny foi o vencedor consistente na validação, alcançando 100% de acurácia nas três seeds e a menor loss de validação. No teste, porém, ResNet18 apresentou a menor loss média e EfficientNet-B0 apresentou o maior F1 médio, ambos com custo muito menor que ConvNeXt-Tiny.

A conclusão não é que um modelo grande sempre vence. O resultado mostra três trade-offs: ConvNeXt-Tiny maximiza validação, ResNet18 apresenta melhor calibração média no teste e EfficientNet-B0 oferece o melhor equilíbrio entre desempenho e custo. Todos superam substancialmente a CNN treinada do zero, reforçando a vantagem de transfer learning em um regime de poucos dados.

A discussão completa, com as tabelas numéricas e limitações metodológicas, está em `docs/final_results.md`.
